# 🚀 Chitragupt — Remote Ollama Server on Colab

Runs **Qwen3-VL 8B** (vision) + **Qwen3 8B** (reasoning) via Ollama
on Colab's free GPU and exposes them via ngrok.

### Persistence

Colab runtimes delete files after ~12 hours. Here's what persists:

| What | Where | Persists? |
|------|-------|-----------|
| Code | GitHub (cloned fresh each session) | ✅ |
| Ollama models (~10GB) | Google Drive `MyDrive/ollama_models/` | ✅ Cached |
| ngrok token | Colab Secrets (🔑) | ✅ Permanent |
| This notebook | Upload to Colab each session, or save to Drive | Optional |

**Why Drive for models?** Models are 10GB — re-downloading every session is impractical.
Drive caches them. Everything else (code, secrets) uses GitHub + Colab Secrets.

---
## 1️⃣ Mount Drive + Clone Code from GitHub

In [ ]:
# @title Mount Google Drive (model cache only)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Clone / pull the Chitragupt repo from GitHub
import os

REPO_PATH = '/content/AI_Chitragupt'
GIT_REPO = 'https://github.com/aditya0701/Chitragupta---A-Vision-based-AI-helper.git'

if os.path.exists(os.path.join(REPO_PATH, '.git')):
    !cd "$REPO_PATH" && git pull
    print('✅ Repo updated from GitHub')
else:
    !git clone "$GIT_REPO" "$REPO_PATH"
    print('✅ Repo cloned from GitHub')

---
## 2️⃣ Setup

In [ ]:
# @title Install Python dependencies
!pip install -q pyngrok nest_asyncio

In [ ]:
# @title Install zstd (required for Ollama)
print('⬇️ Installing zstd...')
!sudo apt-get install -y zstd
print('✅ zstd installed')

In [ ]:
# @title Install Ollama
import os, subprocess, time, logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('chitragupt')

OLLAMA_BIN = '/usr/local/bin/ollama'
if not os.path.exists(OLLAMA_BIN):
    print('⬇️ Installing Ollama...')
    !curl -fsSL https://ollama.com/install.sh | sh
    print('✅ Ollama installed')
else:
    print('✅ Ollama already installed')

!ollama --version

In [ ]:
# @title Setup ngrok (token from Colab Secrets)
from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if NGROK_AUTH_TOKEN:
    !ngrok config add-authtoken $NGROK_AUTH_TOKEN
    print('✅ ngrok authenticated from Colab Secrets')
else:
    print('⚠️  NGROK_AUTH_TOKEN not found in Colab Secrets.')
    print('   Add it: 🔑 (left sidebar) → New Secret → name="NGROK_AUTH_TOKEN", value=your_token')
    print('   This persists across sessions.')

---
## 3️⃣ Start Ollama Server

In [ ]:
# @title Start Ollama server with Google Drive model cache

OLLAMA_MODELS_DIR = '/content/drive/MyDrive/ollama_models'
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
print(f'\U0001f4c1 Model cache: {OLLAMA_MODELS_DIR}')

# Kill zombie + running ollama processes
!kill -9 $(pgrep -f ollama) 2>/dev/null
!pkill -9 -f ollama 2>/dev/null
time.sleep(2)

# Start with Drive cache + CORS + public host (nohup = survives cell interrupts)

!nohup env OLLAMA_MODELS=/content/drive/MyDrive/ollama_models OLLAMA_ORIGINS="*" OLLAMA_HOST="0.0.0.0" ollama serve > /tmp/ollama.log 2>&1 &

time.sleep(3)
!ps aux | grep "ollama serve" | grep -v grep
print('\u2705 Ollama server started on port 11434')


---
## 4️⃣ Pull Models

Models are cached to Google Drive (`MyDrive/ollama_models/`).
**First run only:** downloads ~10GB (10-20 min).
**Subsequent runs:** instant — found on Drive.

In [ ]:
# @title Pull qwen3-vl:8b (Vision model, ~5.5GB)
print('⬇️ Pulling qwen3-vl:8b (vision model)...')
!ollama pull qwen3-vl:8b
print('✅ Vision model ready')

In [ ]:
# @title Pull qwen3:8b (Reasoning model, ~4.6GB)
print('⬇️ Pulling qwen3:8b (reasoning model)...')
!ollama pull qwen3:8b
print('✅ Reasoning model ready')

---
## 5️⃣ Expose via ngrok

In [ ]:
# @title Start ngrok tunnel
from pyngrok import ngrok

ngrok.kill()
tunnel = ngrok.connect(11434, bind_tls=True)
COLAB_OLLAMA_URL = tunnel.public_url

print(f'\n' + '='*60)
print(f'✅ ngrok tunnel established!')
print(f'🌐 Ollama URL: {COLAB_OLLAMA_URL}')
print(f'='*60)

---
## 6️⃣ Validate

In [ ]:
# @title Test both models
import requests
from requests.exceptions import JSONDecodeError

resp = requests.get(f'{COLAB_OLLAMA_URL}/api/tags', timeout=10)
models = [m['name'] for m in resp.json()['models']]
print(f'📦 Models available: {models}')

resp = requests.post(f'{COLAB_OLLAMA_URL}/api/generate', json={
    'model': 'qwen3-vl:8b',
    'prompt': 'Reply with only the word "ready".',
    'stream': False,
}, timeout=300)

try:
    print(f'👁️  Vision model: {resp.json()["response"].strip()}')
except JSONDecodeError as e:
    print(f'Error decoding JSON for vision model: {e}')
    print(f'Raw response text for vision model: {resp.text}')

resp = requests.post(f'{COLAB_OLLAMA_URL}/api/generate', json={
    'model': 'qwen3:8b',
    'prompt': 'Reply with only the word "ready".',
    'stream': False,
}, timeout=300)

try:
    print(f'🧠  Reasoning model: {resp.json()["response"].strip()}')
except JSONDecodeError as e:
    print(f'Error decoding JSON for reasoning model: {e}')
    print(f'Raw response text for reasoning model: {resp.text}')


---
## 7️⃣ Connect Your Local Server

In [ ]:
# @title Connection info → paste into server/.env
print('='*60)
print('         CHITRAGUPT COLAB SERVER — READY')
print('='*60)
print()
print('📋 Copy these into server/.env on your local machine:')
print(f'   BACKEND_MODE=colab')
print(f'   COLAB_OLLAMA_URL={COLAB_OLLAMA_URL}')
print()
print('📝 Quick test from your local machine:')
print(f'   curl {COLAB_OLLAMA_URL}/api/generate -d {{"model":"qwen3-vl:8b","prompt":"Hi","stream":false}}')

In [ ]:
# @title Quick test via Colab itself
import requests
resp = requests.post(f'{COLAB_OLLAMA_URL}/api/chat', json={
    'model': 'qwen3:8b',
    'messages': [{'role': 'user', 'content': 'Say "Chitragupt is ready" in 5 words or less.'}],
    'stream': False,
}, timeout=30)
print('🧠 Chitragupt says:', resp.json()['message']['content'])

---
## 📊 GPU Monitor

In [ ]:
# @title GPU resources
!nvidia-smi --query-gpu=name,memory.used,memory.total,temperature.gpu --format=csv,noheader

---
## 8🤖 Telegram Bot

Runs a Telegram bot directly inside Colab — talks to Ollama at localhost
(no ngrok needed for chat). Chat with Chitragupt via Telegram.

Set your bot token in Colab Secrets: 🔑 → `TELEGRAM_BOT_TOKEN`

Get a token from [@BotFather](https://t.me/botfather) with `/newbot`.

In [ ]:
# @title Install Telegram bot library
!pip install -q python-telegram-bot==21.10 nest_asyncio httpx


In [ ]:
# @title Start Telegram bot
import asyncio, logging
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters
from google.colab import userdata

import nest_asyncio
nest_asyncio.apply()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('chitragupt_bot')

BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
OLLAMA_URL = 'http://localhost:11434'

if not BOT_TOKEN:
    raise SystemExit("\u26a0\ufe0f  Set TELEGRAM_BOT_TOKEN in Colab Secrets")

import httpx

async def chat_with_ollama(prompt: str) -> str:
    payload = {
        'model': 'qwen3:8b',
        'messages': [
            {'role': 'system', 'content': 'You are Chitragupt. Be concise, practical, and helpful.'},
            {'role': 'user', 'content': prompt},
        ],
        'stream': False,
        'options': {'temperature': 0.7, 'num_predict': 2048},
    }
    try:
        async with httpx.AsyncClient(timeout=120.0) as client:
            resp = await client.post(f'{OLLAMA_URL}/api/chat', json=payload)
            resp.raise_for_status()
            return resp.json()['message']['content'].strip()
    except Exception as e:
        return f'Error: {e}'


async def start(update, context):
    await update.message.reply_text("Hello! I'm Chitragupt. Send me a message!")


async def handle_message(update, context):
    prompt = update.message.text
    logger.info(f'Message: {prompt[:50]}...')
    await update.message.chat.send_action(action='typing')
    response = await chat_with_ollama(prompt)
    await update.message.reply_text(response)

# Clear any stale webhook/polling
import httpx as hx
hx.get(f'https://api.telegram.org/bot{BOT_TOKEN}/deleteWebhook?drop_pending_updates=true')

app = Application.builder().token(BOT_TOKEN).build()
app.add_handler(CommandHandler('start', start))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
await app.initialize()
await app.updater.start_polling(drop_pending_updates=True)
await app.start()

print("\U0001f916 Bot is running! Chat with it on Telegram.")
print("(Interrupt cell to stop)")

try:
    while True:
        await asyncio.sleep(1)
except KeyboardInterrupt:
    await app.stop()
    await app.shutdown()


---
## ✅ Next Session

1. Upload this notebook to Colab again (or open from Drive if saved there)
2. **Runtime → Run all**
3. Everything is automatic:
   - ✅ Code cloned fresh from GitHub
   - ✅ Models cached on Drive (no re-download)
   - ✅ ngrok token from Colab Secrets
4. Copy the new ngrok URL → paste in `server/.env`

**Note:** ngrok URL changes every session. Just update `.env`.